In [1]:
import pandas as pd
import os
import pickle
import bmra_prep
import bmra_prep.pathway_activity.prediction

In [2]:
cell_line ='BC3C_dec'

data_dir = f"00_outputs_2020_{cell_line}/"
out_dir = f"01_outputs_2020_{cell_line}/"


os.makedirs(out_dir, exist_ok = True)

# Load Data

In [3]:
out_dir

'01_outputs_2020_BC3C_dec/'

In [4]:
# load metdadata dict and extract used elements
with open(os.path.join(data_dir, "metadata.pickle"), "rb") as f:
    all_metadata = pickle.load(f)

n_modules = all_metadata["n_modules"]
n_genes = all_metadata["n_genes"]
n_experiments = all_metadata["n_experiments"]

modules = all_metadata["modules"]
exp_ids = all_metadata["exp_ids"]
genes = all_metadata["genes"]

In [5]:
# load data
L1000_df = pd.read_csv(
    os.path.join(data_dir, "L1000_Data_norm_data.csv"),
    index_col = 0,
)

x = L1000_df.values
x.shape

(978, 106)

In [6]:
# load doses and perturbation matrix
inhib_conc_matrix = pd.read_csv(
    os.path.join(data_dir, "inhib_conc_annotated.csv"),
    index_col = 0,
).values

ic50_matrix = pd.read_csv(
    os.path.join(data_dir, "ic50_annotated.csv"),
    index_col = 0,
).values

# gamma_matrix = pd.read_csv(
#     os.path.join(data_dir, "gamma_annotated.csv"),
#     index_col = 0,
# ).values

pert_matrix = pd.read_csv(
    os.path.join(data_dir, "pert_annotated.csv"),
    index_col = 0,
).values

In [7]:
# y_true = (1 + gamma_matrix * inhib_conc_matrix / ic50_matrix) / (1 + inhib_conc_matrix / ic50_matrix)

y_true = 1 / (1 + inhib_conc_matrix / ic50_matrix)

display(y_true.shape)
y_true

(10, 106)

array([[1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       ...,
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 0.23076923, 0.23076923,
        0.23076923]], shape=(10, 106))

## Run models

In [8]:
a_coeffs = bmra_prep.pathway_activity.prediction.predict_coeffs(
    x, y_true, pert_matrix, 200_000, 10, 10, 5, 100)

In [9]:
a_coeffs_df = pd.DataFrame(a_coeffs,index=modules,columns = genes)
a_coeffs_df.to_csv(os.path.join(out_dir, "a_coeffs.csv"))
#a_coeffs_df = pd.read_csv('a_coeffs.csv',index_col=0)
#a_coeffs = a_coeffs_df.values
## CHECK how many genes represent the module
trh = 0.0001
display((abs(a_coeffs_df) >trh).sum(axis = "columns"))

# we have too few genes for IFNg, we need to inflate that
display(a_coeffs_df)

CDK1_2      14
CDK4_6       7
EGFR        15
Estrogen    12
FGFR        17
PI3K        36
p53         27
TOP2A       12
Src          7
SMAD3        5
dtype: int64

,AARS,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
CDK1_2,0.000028,1.734024e-05,-1.045944e-05,1.139659e-07,-0.000005,1.041977e-05,2.579659e-06,0.000010,-4.842566e-06,-1.938771e-06,...,-0.000004,-2.556037e-05,1.001266e-05,4.756664e-06,1.505247e-05,-5.378972e-06,-0.000043,6.294312e-07,0.000024,-1.301974e-05
CDK4_6,-0.000011,-7.944304e-06,-1.106423e-05,-5.984227e-06,-0.000005,2.231746e-06,-3.441565e-06,0.000013,3.424820e-05,-2.556238e-05,...,0.000015,-1.907912e-05,1.351065e-05,-1.854312e-05,5.540341e-06,-3.894937e-05,0.000007,-1.760969e-05,-0.000006,-1.342034e-05
EGFR,-0.000013,-3.435697e-05,6.315253e-06,4.592101e-05,-0.000003,-4.592986e-04,-1.116059e-05,0.000022,8.800720e-06,-2.316890e-02,...,-0.000013,-1.193703e-05,6.149973e-06,7.031317e-06,-6.918743e-06,1.336556e-05,0.000007,1.244025e-05,0.000004,9.260778e-07
Estrogen,-0.000028,-4.873306e-06,1.994894e-05,1.896206e-05,-0.000010,5.124092e-08,3.545663e-07,0.000026,-4.677356e-06,-2.993443e-01,...,0.000013,-5.834395e-07,-1.298280e-05,1.846503e-05,-1.104914e-06,-1.891664e-05,-0.000022,-2.528713e-05,-0.000001,1.346762e-05
FGFR,-0.001039,5.681431e-06,3.479437e-07,1.231231e-05,0.000004,-1.798618e-05,-3.191759e-05,-0.000014,-2.268979e-06,-8.781496e-04,...,0.000003,-1.703117e-05,-2.380215e-06,-4.212977e-06,1.525325e-05,2.505495e-04,0.000030,-1.466451e-05,0.000027,-1.369777e-05
PI3K,-0.000004,-1.316439e-06,3.506222e-05,-1.125604e-05,-0.000009,3.672423e-06,-5.357009e-06,-0.000009,-3.290703e-06,-7.996546e-06,...,0.000011,-1.234905e-05,1.258267e-06,-7.667592e-08,1.124168e-05,-2.343848e-05,-0.000010,5.127935e-06,-0.000012,2.884044e-05
p53,-0.000019,-1.498751e-06,-4.858645e-06,8.942766e-07,0.000020,3.576086e-06,2.570168e-01,-0.000037,1.322236e-05,-4.912112e-06,...,0.000024,1.703986e-05,1.565982e-05,-2.441807e-06,2.300614e-05,-2.018618e-05,0.000005,-7.819397e-06,0.000029,3.534313e-05
TOP2A,0.000005,5.514425e-07,-1.447575e-05,-7.079038e-06,0.000010,-2.283768e-05,3.716944e-06,-0.000007,-1.927345e-07,8.676049e-07,...,0.000031,6.069978e-05,7.004550e-06,-3.267329e-05,2.718186e-05,2.703382e-05,0.000020,1.043731e-05,0.000009,-2.960666e-05
Src,-0.000032,5.152189e-05,7.298351e-06,-5.232641e-06,0.000055,1.660646e-06,6.009741e-06,0.000002,2.366338e-06,-3.632627e-05,...,0.000021,-7.855357e-06,2.122266e-05,-7.162805e-06,1.306820e-06,2.019187e-07,-0.000019,1.772477e-06,-0.000002,-1.663072e-05
SMAD3,-0.000012,-2.809872e-05,1.237377e-06,-1.657841e-05,0.000003,-6.793962e-07,2.091102e-05,-0.000002,1.948421e-05,-1.752436e-05,...,0.000021,-5.727761e-06,-4.977549e-07,-1.157944e-05,-1.451912e-07,7.961452e-06,0.000009,1.042817e-05,0.000001,5.798405e-06


In [10]:
#pathway_activity = a_coeffs @ x
#pathway_activity.shape

In [11]:
R_global = bmra_prep.pathway_activity.calc_global_response_from_pathway_activity(
    bmra_prep.pathway_activity.calc_pathway_activity(x,a_coeffs),
    modules,
    L1000_df.columns
)
R_global_df = R_global.dataframe
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD3_V11,SMAD3_V12,SMAD3_V13,SMAD3_V14,SMAD3_V15,SMAD3_V16,SMAD3_V17,SMAD3_V18,SMAD3_V19,SMAD3_V20
CDK1_2,-0.732390,-0.574169,0.050155,0.012087,0.083663,-0.269644,-0.071210,-0.419322,0.168162,0.000455,...,-0.392616,-0.371779,-0.341616,-0.356420,-0.331142,-0.345354,-0.338661,-0.366885,-0.367983,-0.393504
CDK4_6,0.004030,-0.089402,-0.280510,-0.119658,0.028711,-0.109703,-0.095605,-0.044628,-0.255945,0.009910,...,0.242901,0.237699,0.207865,0.187782,0.172492,0.183619,0.212328,0.202168,0.237499,0.206748
EGFR,0.603330,0.512815,0.220783,0.380757,0.550770,0.128326,-0.446151,0.263512,0.283021,-0.022212,...,-1.338967,-0.872734,-0.861788,-0.741935,-0.713491,-0.729167,-0.787042,-0.962566,-0.939140,-0.994961
Estrogen,-0.191659,-0.297802,-0.285529,-0.501957,-0.989677,-0.325297,-0.088943,-0.358269,-0.237886,-0.053961,...,-0.290892,-0.266312,-0.240691,-0.266860,-0.251275,-0.257787,-0.230941,-0.277576,-0.258940,-0.303853
FGFR,-0.117587,-0.197198,-0.100121,0.064252,-0.035630,-0.440261,-0.024733,-0.056993,-0.072518,-0.299410,...,-0.907491,-0.739510,-0.678187,-0.680680,-0.608604,-0.662346,-0.666099,-0.755526,-0.763380,-0.842649
PI3K,-1.938907,-1.866251,-1.679461,-1.422017,-0.626225,-0.173818,-0.007736,-0.480697,-1.360985,-0.234594,...,-0.131956,-0.076407,-0.032055,-0.067374,-0.041878,-0.083457,-0.055419,-0.085363,-0.087070,-0.165278
p53,-0.263693,-0.325991,-0.080932,-0.344558,0.028018,-1.805633,-1.718792,-0.118466,-0.069635,-1.552936,...,0.051077,0.031216,0.044455,0.047418,0.020214,0.047434,0.069707,0.028504,0.070372,0.034327
TOP2A,-0.292815,0.069931,-0.206117,-0.184430,-0.130647,0.020220,0.014743,-1.999184,-0.250836,-0.389917,...,0.153189,0.142490,0.133605,0.112587,0.101270,0.129074,0.139052,0.126773,0.151159,0.145054
Src,-1.059787,-1.997316,0.579319,-1.500732,0.615107,-1.377403,0.538403,0.453884,0.456147,0.498961,...,-0.116252,-0.146028,-0.090411,-0.119875,-0.048745,-0.110704,-0.091114,-0.104768,-0.147164,-0.157167
SMAD3,0.005522,-0.157591,0.044225,-0.404625,0.063312,0.086322,0.034546,0.172641,0.172731,0.059999,...,-1.378356,-1.228306,-1.156636,-1.215284,-1.149212,-1.171070,-1.142051,-1.264353,-1.236354,-1.342972


In [12]:
R_global_df.to_csv(os.path.join(out_dir, "R_global_annotated.csv"))
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD3_V11,SMAD3_V12,SMAD3_V13,SMAD3_V14,SMAD3_V15,SMAD3_V16,SMAD3_V17,SMAD3_V18,SMAD3_V19,SMAD3_V20
CDK1_2,-0.732390,-0.574169,0.050155,0.012087,0.083663,-0.269644,-0.071210,-0.419322,0.168162,0.000455,...,-0.392616,-0.371779,-0.341616,-0.356420,-0.331142,-0.345354,-0.338661,-0.366885,-0.367983,-0.393504
CDK4_6,0.004030,-0.089402,-0.280510,-0.119658,0.028711,-0.109703,-0.095605,-0.044628,-0.255945,0.009910,...,0.242901,0.237699,0.207865,0.187782,0.172492,0.183619,0.212328,0.202168,0.237499,0.206748
EGFR,0.603330,0.512815,0.220783,0.380757,0.550770,0.128326,-0.446151,0.263512,0.283021,-0.022212,...,-1.338967,-0.872734,-0.861788,-0.741935,-0.713491,-0.729167,-0.787042,-0.962566,-0.939140,-0.994961
Estrogen,-0.191659,-0.297802,-0.285529,-0.501957,-0.989677,-0.325297,-0.088943,-0.358269,-0.237886,-0.053961,...,-0.290892,-0.266312,-0.240691,-0.266860,-0.251275,-0.257787,-0.230941,-0.277576,-0.258940,-0.303853
FGFR,-0.117587,-0.197198,-0.100121,0.064252,-0.035630,-0.440261,-0.024733,-0.056993,-0.072518,-0.299410,...,-0.907491,-0.739510,-0.678187,-0.680680,-0.608604,-0.662346,-0.666099,-0.755526,-0.763380,-0.842649
PI3K,-1.938907,-1.866251,-1.679461,-1.422017,-0.626225,-0.173818,-0.007736,-0.480697,-1.360985,-0.234594,...,-0.131956,-0.076407,-0.032055,-0.067374,-0.041878,-0.083457,-0.055419,-0.085363,-0.087070,-0.165278
p53,-0.263693,-0.325991,-0.080932,-0.344558,0.028018,-1.805633,-1.718792,-0.118466,-0.069635,-1.552936,...,0.051077,0.031216,0.044455,0.047418,0.020214,0.047434,0.069707,0.028504,0.070372,0.034327
TOP2A,-0.292815,0.069931,-0.206117,-0.184430,-0.130647,0.020220,0.014743,-1.999184,-0.250836,-0.389917,...,0.153189,0.142490,0.133605,0.112587,0.101270,0.129074,0.139052,0.126773,0.151159,0.145054
Src,-1.059787,-1.997316,0.579319,-1.500732,0.615107,-1.377403,0.538403,0.453884,0.456147,0.498961,...,-0.116252,-0.146028,-0.090411,-0.119875,-0.048745,-0.110704,-0.091114,-0.104768,-0.147164,-0.157167
SMAD3,0.005522,-0.157591,0.044225,-0.404625,0.063312,0.086322,0.034546,0.172641,0.172731,0.059999,...,-1.378356,-1.228306,-1.156636,-1.215284,-1.149212,-1.171070,-1.142051,-1.264353,-1.236354,-1.342972
